In [ ]:
# Bật GPU (T4/P100) và Internet trong Settings trước khi chạy
!git clone https://github.com/hotuyen21pt/MultiLABSA.git

# Kaggle đã có sẵn torch/transformers/pandas/numpy/tqdm — chỉ cài thứ còn thiếu,
# KHÔNG cài lại torch để tránh phá bản CUDA của Kaggle. Danh sách requirement
# nam trong dapt/requirements-kaggle.txt (sentencepiece, protobuf, va
# wordfreq[cjk] + fasttext-wheel cho build_lexicon.py).
!pip install -q -r MultiLABSA/dapt/requirements-kaggle.txt

In [ ]:
import os, fnmatch, torch

# Kaggle mount cac dataset duoi /kaggle/input bang symlink toi thu muc phien ban
# (vi du /kaggle/input/datasets/<owner>/<slug>/versions/N/...), nen glob("**") va
# os.walk mac dinh (followlinks=False) KHONG di xuyen duoc qua symlink va bao
# "khong tim thay file" du dataset da duoc Add Input dung cach.
# => phai dung os.walk(..., followlinks=True) de duyet xuyen qua symlink.
PATTERNS = ("hotel_review*.csv", "*_lang.csv")

matches = []
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    for fn in files:
        if any(fnmatch.fnmatch(fn, p) for p in PATTERNS):
            matches.append(os.path.join(root, fn))

if not matches:
    print("Khong tim thay hotel_review*.csv / *_lang.csv nao duoi /kaggle/input.")
    print("Cay thu muc duoi /kaggle/input (theo symlink, toi da 5 cap):")
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
        if depth > 5:
            dirs[:] = []
            continue
        print("  " * depth + os.path.basename(root) + "/")
        for fn in files[:5]:
            print("  " * (depth + 1) + fn)
    raise FileNotFoundError(
        "Khong tim thay corpus (hotel_review*.csv hoac *_lang.csv) duoi /kaggle/input. "
        "Kiem tra da them dataset trong Settings > Add Input (xem cay thu muc in o tren)."
    )

SRC_DIR = os.path.dirname(matches[0])
print("SRC_DIR =", SRC_DIR)
print("Files:", os.listdir(SRC_DIR))

print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("bf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

In [ ]:
import glob, os, subprocess, urllib.request

existing_lang = glob.glob(os.path.join(SRC_DIR, "*_lang.csv"))
if existing_lang:
    DATA = SRC_DIR
    print("SRC_DIR da co san file *_lang.csv, dung truc tiep:", DATA)
else:
    LANG_OUT = "/kaggle/working/data_final/unlabeled_data"
    MODEL_PATH = "/kaggle/working/lid.176.bin"
    os.makedirs(LANG_OUT, exist_ok=True)

    already_done = glob.glob(os.path.join(LANG_OUT, "*_lang.csv"))
    if already_done:
        print("Da chay add_language truoc do, bo qua:", LANG_OUT)
    else:
        if not os.path.exists(MODEL_PATH):
            print("Tai model fastText lid.176.bin (~126MB)...")
            urllib.request.urlretrieve(
                "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin",
                MODEL_PATH,
            )
        subprocess.run([
            "python", "/kaggle/working/MultiLABSA/add_language.py",
            "--model_path", MODEL_PATH,
            "--unlabeled_dir", SRC_DIR,
            "--out_dir", LANG_OUT,
            "--skip_labeled",
        ], check=True)

    DATA = LANG_OUT

print("DATA (co cot language) =", DATA)
print("Files:", os.listdir(DATA))

In [ ]:
%cd /kaggle/working/MultiLABSA/dapt
!python build_lexicon.py \
    --data_dir {DATA} \
    --languages en vi fr de es it nl ru ja ko zh \
    --max_per_lang 200000 --top_k 300 --top_k_bigrams 100 \
    --min_general_freq 1e-6 --expand \
    --out_dir /kaggle/working/lexicon

In [ ]:
%cd /kaggle/working/MultiLABSA/dapt
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# Van OOM ngay trong optimizer.step() du batch_size=2 + gradient_checkpointing:
# model + gradients + AdamW state (2 moment fp32 rieng) da chiem ~14.44/14.56 GiB
# TRUOC ca activation - nghia la ban than optimizer state qua lon cho GPU nay.
# --optimizer adafactor dung second moment factored (khong luu first moment day
# du) nen state nho hon AdamW rat nhieu, day la optimizer T5/mT5 goc dung khi
# pretrain vi ly do nay.
!python train_dapt.py \
    --data_dir {DATA} \
    --lexicon_file /kaggle/working/lexicon/lexicon.json \
    --precision auto \
    --optimizer adafactor \
    --batch_size 2 --gradient_accumulation_steps 16 \
    --gradient_checkpointing \
    --num_epochs 1 --max_steps 5000 \
    --warmup_steps 500 --eval_steps 500 --save_steps 500 \
    --output_dir /kaggle/working/checkpoints \
    --final_dir /kaggle/working/hotel-mt5

In [ ]:
%cd /kaggle/working/MultiLABSA/dapt
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# --resume chi khoi phuc optimizer/scheduler/scaler tu training_state.pt, KHONG
# luu lai batch_size/gradient_checkpointing/optimizer - phai truyen lai giong
# cell truoc moi lan chay (--optimizer phai khop voi lan train truoc, vi
# optimizer.load_state_dict se sai neu doi loai optimizer giua chung).
!python train_dapt.py --resume \
    --data_dir {DATA} \
    --lexicon_file /kaggle/working/lexicon/lexicon.json \
    --optimizer adafactor \
    --batch_size 2 --gradient_accumulation_steps 16 \
    --gradient_checkpointing \
    --output_dir /kaggle/working/checkpoints \
    --final_dir /kaggle/working/hotel-mt5

In [6]:
import shutil
shutil.make_archive("/kaggle/working/hotel-mt5", "zip", "/kaggle/working/hotel-mt5")
print("Saved /kaggle/working/hotel-mt5.zip")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/hotel-mt5'